# 02 -- Train a Single Configuration

**Environment: Google Colab (T4 GPU)**

| What | Where |
|---|---|
| Source code | `/content/drive/MyDrive/Ai_TRAINING/COMP2050_Project` |
| Dataset | `.../data/Kvasir-SEG/` on Drive |
| Checkpoints & logs | `.../results/` on Drive (synced every 5 epochs) |

**Prerequisites:** Run `01_setup_data.ipynb` first.

**Steps:**
1. Mount Drive & load config
2. Train
3. Visualize training curves
4. Visualize segmentation results
5. Visualize attention maps (attention_unet only)

---
## 0 - Mount Drive & setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import torch

REPO_ROOT  = Path("/content/drive/MyDrive/Ai_TRAINING/COMP2050_Project")
DRIVE_DATA = REPO_ROOT / "data"
DRIVE_RESULTS = REPO_ROOT / "results"

sys.path.insert(0, str(REPO_ROOT))

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# Configuration - change these to run different experiments
ARCHITECTURE = 'attention_unet'  # unet | attention_unet | resunet | smp_resnet18
LOSS = 'bce_dice'                # bce | dice | focal | bce_dice | tversky
SEED = 42
EPOCHS = 50
LR = 1e-4
BATCH_SIZE = 16
PATIENCE = 10

print(f"Config: {ARCHITECTURE} | {LOSS} | seed={SEED}")

---
## 1 - Train

In [ ]:
from train import train

config = {
    'architecture': ARCHITECTURE,
    'loss': LOSS,
    'seed': SEED,
    'epochs': EPOCHS,
    'lr': LR,
    'batch_size': BATCH_SIZE,
    'patience': PATIENCE,
    'data_root': str(DRIVE_DATA),
    'output_dir': str(DRIVE_RESULTS),
}

results = train(config)
print(f"\nTest Dice: {results['test_dice']:.4f}")

---
## 2 - Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

run_name = f'{ARCHITECTURE}_{LOSS}_seed{SEED}'
with open(DRIVE_RESULTS / run_name / 'history.json') as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title(f'Training Loss ({run_name})'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['val_dice'], label='Val Dice', color='green')
ax2.plot(history['val_iou'], label='Val IoU', color='blue')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score')
ax2.set_title(f'Validation Metrics ({run_name})'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

---
## 3 - Segmentation results

In [ ]:
import numpy as np
from models import get_model
from data.dataset import KvasirSEGDataset

# Load best model
model = get_model(ARCHITECTURE)
model.load_state_dict(torch.load(
    DRIVE_RESULTS / run_name / 'best_model.pth', weights_only=True
))
model.eval().cuda()

test_dataset = KvasirSEGDataset(str(DRIVE_DATA), split='test', seed=SEED)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
indices = [0, len(test_dataset)//2, len(test_dataset)-1]

for row, idx in enumerate(indices):
    img, mask = test_dataset[idx]
    with torch.no_grad():
        pred = model(img.unsqueeze(0).cuda())
        pred = torch.sigmoid(pred).squeeze().cpu().numpy()

    img_np = img.numpy().transpose(1, 2, 0)
    img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    mask_np = mask.squeeze().numpy()
    pred_bin = (pred > 0.5).astype(np.float32)

    axes[row, 0].imshow(img_np); axes[row, 0].set_title('Input')
    axes[row, 1].imshow(mask_np, cmap='gray'); axes[row, 1].set_title('Ground Truth')
    axes[row, 2].imshow(pred, cmap='gray'); axes[row, 2].set_title('Prediction (prob)')
    axes[row, 3].imshow(img_np); axes[row, 3].imshow(pred_bin, cmap='Reds', alpha=0.4)
    axes[row, 3].set_title('Overlay')
    for c in range(4): axes[row, c].axis('off')

plt.suptitle(f'Segmentation: {run_name}', fontsize=14); plt.tight_layout(); plt.show()

---
## 4 - Attention maps (attention_unet only)

In [ ]:
if ARCHITECTURE == 'attention_unet':
    from utils.visualization import plot_attention_maps

    idx = 0
    img, mask = test_dataset[idx]
    with torch.no_grad():
        _ = model(img.unsqueeze(0).cuda())
        att_maps = model.get_attention_maps()

    plot_attention_maps(img, att_maps)
else:
    print('Attention maps only available for attention_unet.')